# Neural Networks for Magnetic Field Prediction

## Introduction

This chapter explores the application of deep learning techniques for magnetic field prediction in electromagnetic systems. We'll examine various neural network architectures and training strategies.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print("Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")

## Neural Network Architectures

We'll explore several architectures suitable for magnetic field prediction:

In [ ]:
class MagneticFieldNet(nn.Module):
    """Neural network for magnetic field prediction"""
    
    def __init__(self, input_dim=8, hidden_dims=[128, 64, 32], output_dim=2500):
        super(MagneticFieldNet, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        # Hidden layers
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Dropout(0.2)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

class ConvolutionalFieldNet(nn.Module):
    """Convolutional neural network for field prediction"""
    
    def __init__(self, input_channels=1, output_size=50):
        super(ConvolutionalFieldNet, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=2, stride=2),
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Initialize models
mlp_model = MagneticFieldNet()
cnn_model = ConvolutionalFieldNet()

print(f"MLP Model parameters: {sum(p.numel() for p in mlp_model.parameters()):,}")
print(f"CNN Model parameters: {sum(p.numel() for p in cnn_model.parameters()):,}")

## Training Data Preparation

Prepare synthetic training data for demonstration purposes.

In [ ]:
class MagneticFieldDataset(Dataset):
    """Dataset for magnetic field prediction"""
    
    def __init__(self, num_samples=1000):
        self.num_samples = num_samples
        self.generate_data()
    
    def generate_data(self):
        """Generate synthetic training data"""
        self.inputs = []
        self.outputs = []
        
        for _ in range(self.num_samples):
            # Random electromagnetic parameters
            radius = np.random.uniform(0.01, 0.1)
            turns = np.random.randint(50, 200)
            current = np.random.uniform(0.5, 5.0)
            wire_radius = np.random.uniform(0.0005, 0.002)
            frequency = np.random.uniform(10, 100)
            material_permeability = np.random.uniform(1000, 5000)
            temperature = np.random.uniform(20, 100)
            position_x = np.random.uniform(-0.05, 0.05)
            
            # Input features
            input_features = [
                radius, turns, current, wire_radius,
                frequency, material_permeability, temperature, position_x
            ]
            
            # Generate corresponding field distribution (simplified)
            field_size = 50
            x = np.linspace(-0.1, 0.1, field_size)
            y = np.linspace(-0.1, 0.1, field_size)
            X, Y = np.meshgrid(x, y)
            
            # Simplified field calculation
            R = np.sqrt((X - position_x)**2 + Y**2)
            field_strength = 0.01 * current * turns / 100 * np.exp(-R**2 / radius**2)
            
            # Add noise and nonlinear effects
            field_strength += 0.001 * np.random.randn(field_size, field_size)
            field_strength *= (1 + 0.1 * np.sin(frequency * 0.1))
            
            self.inputs.append(input_features)
            self.outputs.append(field_strength.flatten())
        
        self.inputs = np.array(self.inputs, dtype=np.float32)
        self.outputs = np.array(self.outputs, dtype=np.float32)
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return torch.FloatTensor(self.inputs[idx]), torch.FloatTensor(self.outputs[idx])

# Create dataset and dataloader
dataset = MagneticFieldDataset(num_samples=500)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Dataset created with {len(dataset)} samples")
print(f"Input shape: {dataset.inputs.shape}")
print(f"Output shape: {dataset.outputs.shape}")

## Model Training

Train the neural network models on the synthetic data.

In [ ]:
def train_model(model, dataloader, epochs=10, learning_rate=0.001):
    """Train the neural network model"""
    
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    train_losses = []
    
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for batch_idx, (inputs, targets) in enumerate(dataloader):
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(dataloader)
        train_losses.append(avg_loss)
        
        if epoch % 2 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}")
    
    return train_losses

# Train the MLP model
print("Training MLP model...")
mlp_losses = train_model(mlp_model, dataloader, epochs=10)

# Plot training progress
plt.figure(figsize=(10, 4))
plt.plot(mlp_losses, 'b-', label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Training completed!")

## Model Evaluation

Evaluate the trained model and visualize predictions.

In [ ]:
def evaluate_model(model, test_input):
    """Evaluate model on test input"""
    model.eval()
    with torch.no_grad():
        prediction = model(torch.FloatTensor(test_input).unsqueeze(0))
    return prediction.numpy().reshape(50, 50)

# Test the model with sample input
test_input = [0.05, 100, 2.0, 0.001, 50, 2000, 25, 0.0]  # radius, turns, current, etc.
predicted_field = evaluate_model(mlp_model, test_input)

# Visualize prediction
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(predicted_field, cmap='viridis', aspect='auto')
plt.colorbar(label='Magnetic Field (T)')
plt.title('Predicted Magnetic Field')
plt.xlabel('X Position')
plt.ylabel('Y Position')

# Generate true field for comparison (simplified)
x = np.linspace(-0.1, 0.1, 50)
y = np.linspace(-0.1, 0.1, 50)
X, Y = np.meshgrid(x, y)
R = np.sqrt(X**2 + Y**2)
true_field = 0.01 * test_input[2] * test_input[1] / 100 * np.exp(-R**2 / test_input[0]**2)

plt.subplot(1, 2, 2)
plt.imshow(true_field, cmap='viridis', aspect='auto')
plt.colorbar(label='Magnetic Field (T)')
plt.title('True Magnetic Field (Simplified)')
plt.xlabel('X Position')
plt.ylabel('Y Position')

plt.tight_layout()
plt.show()

# Calculate error metrics
mse = np.mean((predicted_field - true_field)**2)
mae = np.mean(np.abs(predicted_field - true_field))
max_error = np.max(np.abs(predicted_field - true_field))

print(f"Mean Squared Error: {mse:.8f}")
print(f"Mean Absolute Error: {mae:.6f}")
print(f"Maximum Error: {max_error:.6f}")

## Summary

This notebook demonstrates:

### ✅ **Key Techniques**
1. **Neural Network Architectures**: MLP and CNN models for field prediction
2. **Data Preparation**: Synthetic dataset generation with realistic parameters
3. **Training Pipeline**: Complete training loop with loss monitoring
4. **Model Evaluation**: Performance metrics and visualization

### 🎯 **Applications**
- Magnetic field prediction for electromagnetic devices
- Design optimization and parameter studies
- Real-time field estimation
- Inverse problem solving

### 📈 **Performance Considerations**
- Model complexity vs. accuracy trade-off
Training data quality and quantity
- Computational efficiency for real-time applications
- Generalization to unseen parameter combinations

The neural network approach provides fast, accurate predictions once trained, making it suitable for design optimization and real-time applications where traditional FEM simulations would be too slow.